# 90. The Global Sourcing Offshoring vs. Reshoring Problem
## Tier 1: The Pen & Paper Method (Mathematical Formulation)

### Key assumptions
- Multiple products/components can be sourced from multiple locations
- Each location has different cost, risk, quality, and capacity characteristics
- Demand must be satisfied for each product in each time period
- Risk diversification limits exposure to high-risk locations
- Fixed costs are incurred when sourcing from any location

### Approach (step-by-step)
1. **Define decision variables**: Quantity of each product sourced from each location in each time period
2. **Formulate objective function**: Multi-objective optimization balancing cost, risk, and quality
3. **Add constraints**: Demand satisfaction, capacity limits, risk diversification, and binary activation
4. **Solve using convex optimization**: Find optimal sourcing allocation
5. **Analyze results**: Extract optimal solution and perform sensitivity analysis

### What to look for in the results
- Optimal allocation matrix showing how much of each product to source from each location
- Trade-off between cost efficiency and risk mitigation
- Impact of quality bonuses on sourcing decisions
- Risk diversification constraints affecting the solution

### Concrete example (from the source)
GlobalTech Industries sources three components (microprocessors, displays, batteries) from four locations (China, Mexico, US, hybrid) with:
- Monthly demands: 10,000 microprocessors, 8,000 displays, 12,000 batteries
- Unit costs: China $25, Mexico $35, US $50
- Risk factors: China 0.8, Mexico 0.4, US 0.1
- Quality scores: China 0.7, Mexico 0.8, US 0.95
- Lead times: China 8 weeks, Mexico 3 weeks, US 1 week

### Why this Tier exists vs earlier Tiers
This is the foundational Tier that provides the mathematical formulation for the global sourcing problem. It establishes:
- The theoretical framework for multi-objective optimization
- Baseline optimal solutions for comparison with heuristic methods
- Understanding of the trade-offs between cost, risk, and quality
- Mathematical foundation for all subsequent solution approaches

### Pros / Cons vs earlier Tiers
**Pros:**
- Provides provably optimal solutions
- Clear mathematical understanding of the problem structure
- Comprehensive sensitivity analysis capabilities
- Transparent objective function and constraints

**Cons:**
- Computationally intensive for large problem instances
- Requires precise parameter estimation
- May not scale well to real-world complexity
- Limited flexibility for dynamic changes

### When to use this Tier
- Small to medium-sized sourcing problems with complete data
- Strategic planning where optimality is critical
- Benchmark setting for heuristic evaluation
- Situations requiring detailed sensitivity analysis

In [1]:
# Import required libraries for mathematical optimization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Set style for professional visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [2]:
# Define data structures for the sourcing problem
class SourcingData:
    """Data structure for global sourcing optimization problem"""
    def __init__(self):
        # Products and their monthly demands
        self.products = ['Microprocessor', 'Display', 'Battery']
        self.demands = {'Microprocessor': 10000, 'Display': 8000, 'Battery': 12000}
        
        # Sourcing locations and their characteristics
        self.locations = ['China', 'Mexico', 'US']
        self.unit_costs = {'China': 25, 'Mexico': 35, 'US': 50}
        self.risk_factors = {'China': 0.8, 'Mexico': 0.4, 'US': 0.1}
        self.quality_scores = {'China': 0.7, 'Mexico': 0.8, 'US': 0.95}
        self.lead_times = {'China': 8, 'Mexico': 3, 'US': 1}  # weeks
        self.capacities = {'China': 50000, 'Mexico': 30000, 'US': 25000}
        
        # Fixed costs for sourcing from each location
        self.fixed_costs = {'China': 10000, 'Mexico': 8000, 'US': 12000}
        
        # Objective function weights
        self.alpha = 0.6  # Cost weight
        self.beta = 0.3   # Risk weight
        self.gamma = 0.1  # Quality weight
        
        # Risk diversification parameters
        self.high_risk_threshold = 0.6
        self.max_high_risk_percentage = 0.6

# Initialize the problem data
data = SourcingData()
print(f"Products: {data.products}")
print(f"Locations: {data.locations}")
print(f"Total monthly demand: {sum(data.demands.values()):,}")

Products: ['Microprocessor', 'Display', 'Battery']
Locations: ['China', 'Mexico', 'US']
Total monthly demand: 30,000


Products: ['Microprocessor', 'Display', 'Battery']
Locations: ['China', 'Mexico', 'US']
Total monthly demand: 30,000


Products: ['Microprocessor', 'Display', 'Battery']
Locations: ['China', 'Mexico', 'US']
Total monthly demand: 30,000


Products: ['Microprocessor', 'Display', 'Battery']
Locations: ['China', 'Mexico', 'US']
Total monthly demand: 30,000


Products: ['Microprocessor', 'Display', 'Battery']
Locations: ['China', 'Mexico', 'US']
Total monthly demand: 30,000


In [3]:
def objective_function(x_flat, data):
    """Multi-objective function for global sourcing optimization
    
    Args:
        x_flat: Flattened decision variables [x_ijt for all i,j,t]
        data: SourcingData object containing problem parameters
    
    Returns:
        Total objective value (cost + risk penalty - quality bonus)
    """
    # Reshape flat array to 3D matrix (products x locations x time_periods)
    # For this example, we use 1 time period
    n_products = len(data.products)
    n_locations = len(data.locations)
    n_periods = 1  # Single period optimization
    
    x = x_flat.reshape(n_products, n_locations, n_periods)
    
    # Calculate total cost component
    total_cost = 0
    for i, product in enumerate(data.products):
        for j, location in enumerate(data.locations):
            unit_cost = data.unit_costs[location]
            quantity = x[i, j, 0]
            # Add variable cost
            total_cost += unit_cost * quantity
            # Add fixed cost if location is used
            if quantity > 0:
                total_cost += data.fixed_costs[location]
    
    # Calculate risk penalty
    risk_penalty = 0
    for i, product in enumerate(data.products):
        for j, location in enumerate(data.locations):
            risk_factor = data.risk_factors[location]
            quantity = x[i, j, 0]
            risk_penalty += risk_factor * quantity
    
    # Calculate quality bonus (negative because we maximize quality)
    quality_bonus = 0
    for i, product in enumerate(data.products):
        for j, location in enumerate(data.locations):
            quality_score = data.quality_scores[location]
            quantity = x[i, j, 0]
            quality_bonus += quality_score * quantity
    
    # Combine objectives with weights
    total_objective = (data.alpha * total_cost + 
                      data.beta * risk_penalty - 
                      data.gamma * quality_bonus)
    
    return total_objective

In [4]:
def constraint_demands(x_flat, data):
    """Demand satisfaction constraints"""
    n_products = len(data.products)
    n_locations = len(data.locations)
    x = x_flat.reshape(n_products, n_locations, 1)
    
    constraints = []
    for i, product in enumerate(data.products):
        demand = data.demands[product]
        total_sourced = np.sum(x[i, :, 0])
        constraints.append(total_sourced - demand)  # Should equal 0
    
    return constraints

def constraint_capacities(x_flat, data):
    """Capacity constraints"""
    n_products = len(data.products)
    n_locations = len(data.locations)
    x = x_flat.reshape(n_products, n_locations, 1)
    
    constraints = []
    for j, location in enumerate(data.locations):
        capacity = data.capacities[location]
        total_used = np.sum(x[:, j, 0])
        constraints.append(capacity - total_used)  # Should be >= 0
    
    return constraints

def constraint_risk_diversification(x_flat, data):
    """Risk diversification constraints"""
    n_products = len(data.products)
    n_locations = len(data.locations)
    x = x_flat.reshape(n_products, n_locations, 1)
    
    constraints = []
    for i, product in enumerate(data.products):
        # Identify high-risk locations
        high_risk_indices = [j for j, loc in enumerate(data.locations) 
                           if data.risk_factors[loc] >= data.high_risk_threshold]
        
        if high_risk_indices:
            high_risk_total = np.sum([x[i, j, 0] for j in high_risk_indices])
            total_sourced = np.sum(x[i, :, 0])
            max_allowed = data.max_high_risk_percentage * total_sourced
            constraints.append(max_allowed - high_risk_total)  # Should be >= 0
    
    return constraints

In [ ]:
# Set up and solve the optimization problem
n_products = len(data.products)
n_locations = len(data.locations)
n_variables = n_products * n_locations

# Initial guess: distribute demand proportionally
x0 = np.zeros(n_variables)
for i, product in enumerate(data.products):
    demand = data.demands[product]
    for j, location in enumerate(data.locations):
        # Initial allocation proportional to capacity
        total_capacity = sum(data.capacities.values())
        proportion = data.capacities[location] / total_capacity
        x0[i * n_locations + j] = demand * proportion

# Bounds: non-negative quantities
bounds = [(0, None) for _ in range(n_variables)]

# Constraints
constraints = []

# Demand satisfaction constraints (equality)
for i in range(n_products):
    def demand_constraint(x_flat, idx=i):
        return constraint_demands(x_flat, data)[idx]
    
    constraints.append({'type': 'eq', 'fun': demand_constraint})

# Capacity constraints (inequality)
for j in range(n_locations):
    def capacity_constraint(x_flat, idx=j):
        return constraint_capacities(x_flat, data)[idx]
    
    constraints.append({'type': 'ineq', 'fun': capacity_constraint})

# Risk diversification constraints (inequality)
risk_constraints = constraint_risk_diversification(x0, data)
for k in range(len(risk_constraints)):
    def risk_constraint(x_flat, idx=k):
        return constraint_risk_diversification(x_flat, data)[idx]
    
    constraints.append({'type': 'ineq', 'fun': risk_constraint})

print(f"Number of decision variables: {n_variables}")
print(f"Number of constraints: {len(constraints)}")
print(f"Initial total allocation: {np.sum(x0):,}")

In [ ]:
# Solve the optimization problem
print("Solving global sourcing optimization problem...")

result = minimize(
    objective_function,
    x0,
    args=(data,),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints,
    options={'maxiter': 1000, 'ftol': 1e-9}
)

if result.success:
    print("✓ Optimization successful!")
    print(f"Objective value: ${result.fun:,.2f}")
    print(f"Iterations: {result.nit}")
    print(f"Function evaluations: {result.nfev}")
else:
    print("✗ Optimization failed:", result.message)

In [ ]:
# Extract and analyze the optimal solution
if result.success:
    # Reshape solution to matrix format
    optimal_allocation = result.x.reshape(n_products, n_locations, 1)
    
    # Create results DataFrame
    results_df = pd.DataFrame(
        optimal_allocation[:,:,0].round(0),
        index=data.products,
        columns=data.locations
    )
    
    # Add totals
    results_df['Total Demand'] = [data.demands[p] for p in data.products]
    results_df.loc['Total Allocation'] = results_df.iloc[:-1].sum(axis=0)
    
    print("=== OPTIMAL SOURCING ALLOCATION ===")
    print(results_df.to_string())
    print()
    
    # Calculate detailed cost breakdown
    total_variable_cost = 0
    total_fixed_cost = 0
    total_risk_penalty = 0
    total_quality_bonus = 0
    
    print("=== COST BREAKDOWN ===")
    for j, location in enumerate(data.locations):
        location_quantity = np.sum(optimal_allocation[:, j, 0])
        if location_quantity > 0:
            variable_cost = data.unit_costs[location] * location_quantity
            fixed_cost = data.fixed_costs[location]
            risk_penalty = data.risk_factors[location] * location_quantity
            quality_bonus = data.quality_scores[location] * location_quantity
            
            total_variable_cost += variable_cost
            total_fixed_cost += fixed_cost
            total_risk_penalty += risk_penalty
            total_quality_bonus += quality_bonus
            
            print(f"{location}:")
            print(f"  Quantity: {location_quantity:,.0f} units")
            print(f"  Variable cost: ${variable_cost:,.2f}")
            print(f"  Fixed cost: ${fixed_cost:,.2f}")
            print(f"  Risk penalty: {risk_penalty:,.0f}")
            print(f"  Quality bonus: {quality_bonus:,.0f}")
            print()
    
    # Calculate total objective components
    cost_component = data.alpha * (total_variable_cost + total_fixed_cost)
    risk_component = data.beta * total_risk_penalty
    quality_component = data.gamma * total_quality_bonus
    
    print("=== OBJECTIVE FUNCTION BREAKDOWN ===")
    print(f"Total variable cost: ${total_variable_cost:,.2f}")
    print(f"Total fixed cost: ${total_fixed_cost:,.2f}")
    print(f"Total risk penalty: {total_risk_penalty:,.0f}")
    print(f"Total quality bonus: {total_quality_bonus:,.0f}")
    print()
    print(f"Cost component (α={data.alpha}): ${cost_component:,.2f}")
    print(f"Risk component (β={data.beta}): ${risk_component:,.2f}")
    print(f"Quality component (γ={data.gamma}): ${quality_component:,.2f}")
    print(f"Total objective: ${result.fun:,.2f}")

In [ ]:
# Create comprehensive visualization
if result.success:
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Global Sourcing Optimization Analysis', fontsize=16, fontweight='bold')
    
    # 1. Allocation heatmap
    ax1 = axes[0, 0]
    allocation_matrix = optimal_allocation[:,:,0]
    sns.heatmap(allocation_matrix, 
                annot=True, 
                fmt='.0f',
                xticklabels=data.locations,
                yticklabels=data.products,
                cmap='Blues',
                ax=ax1)
    ax1.set_title('Optimal Sourcing Allocation')
    ax1.set_xlabel('Sourcing Locations')
    ax1.set_ylabel('Products')
    
    # 2. Cost breakdown by location
    ax2 = axes[0, 1]
    location_costs = []
    location_names = []
    
    for j, location in enumerate(data.locations):
        quantity = np.sum(optimal_allocation[:, j, 0])
        if quantity > 0:
            total_cost = (data.unit_costs[location] * quantity + 
                         data.fixed_costs[location])
            location_costs.append(total_cost)
            location_names.append(location)
    
    bars = ax2.bar(location_names, location_costs, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
    ax2.set_title('Total Cost by Sourcing Location')
    ax2.set_xlabel('Sourcing Location')
    ax2.set_ylabel('Total Cost ($)')
    ax2.tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for bar, cost in zip(bars, location_costs):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(location_costs)*0.01,
                f'${cost:,.0f}', ha='center', va='bottom')
    
    # 3. Risk vs Quality scatter plot
    ax3 = axes[1, 0]
    for j, location in enumerate(data.locations):
        quantity = np.sum(optimal_allocation[:, j, 0])
        if quantity > 0:
            ax3.scatter(data.risk_factors[location], 
                      data.quality_scores[location],
                      s=quantity/100, 
                      alpha=0.7,
                      label=location)
    
    ax3.set_xlabel('Risk Factor')
    ax3.set_ylabel('Quality Score')
    ax3.set_title('Risk-Quality Trade-off (Size = Volume)')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 4. Objective function components
    ax4 = axes[1, 1]
    components = ['Cost\nComponent', 'Risk\nComponent', 'Quality\nComponent']
    values = [cost_component, risk_component, -quality_component]  # Negative for quality since it's a bonus
    colors = ['#FF6B6B', '#FFA07A', '#98D8C8']
    
    bars = ax4.bar(components, values, color=colors)
    ax4.set_title('Objective Function Components')
    ax4.set_ylabel('Contribution to Objective ($)')
    ax4.axhline(y=0, color='black', linestyle='-', alpha=0.3)
    ax4.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bar, value in zip(bars, values):
        ax4.text(bar.get_x() + bar.get_width()/2, 
                bar.get_height() + (max(values) - min(values))*0.01,
                f'${value:,.0f}', ha='center', va='bottom' if value >= 0 else 'top')
    
    plt.tight_layout()
    plt.show()
else:
    print("Cannot create visualizations - optimization failed")

In [ ]:
# Sensitivity analysis on objective function weights
if result.success:
    print("=== SENSITIVITY ANALYSIS ===")
    print("\nAnalyzing impact of different weight combinations...")
    
    # Test different weight scenarios
    weight_scenarios = [
        {'name': 'Cost-focused', 'alpha': 0.8, 'beta': 0.15, 'gamma': 0.05},
        {'name': 'Balanced', 'alpha': 0.6, 'beta': 0.3, 'gamma': 0.1},
        {'name': 'Risk-averse', 'alpha': 0.4, 'beta': 0.5, 'gamma': 0.1},
        {'name': 'Quality-focused', 'alpha': 0.5, 'beta': 0.2, 'gamma': 0.3}
    ]
    
    sensitivity_results = []
    
    for scenario in weight_scenarios:
        # Update weights
        data.alpha = scenario['alpha']
        data.beta = scenario['beta']
        data.gamma = scenario['gamma']
        
        # Re-solve with new weights
        try:
            result_sensitivity = minimize(
                objective_function,
                x0,  # Use same initial guess
                args=(data,),
                method='SLSQP',
                bounds=bounds,
                constraints=constraints,
                options={'maxiter': 500}
            )
            
            if result_sensitivity.success:
                # Calculate allocation distribution
                allocation = result_sensitivity.x.reshape(n_products, n_locations, 1)
                china_pct = np.sum(allocation[:, 0, 0]) / np.sum(allocation) * 100
                mexico_pct = np.sum(allocation[:, 1, 0]) / np.sum(allocation) * 100
                us_pct = np.sum(allocation[:, 2, 0]) / np.sum(allocation) * 100
                
                sensitivity_results.append({
                    'Scenario': scenario['name'],
                    'Objective': result_sensitivity.fun,
                    'China %': china_pct,
                    'Mexico %': mexico_pct,
                    'US %': us_pct
                })
        except Exception as e:
            print(f"Error in scenario {scenario['name']}: {e}")
    
    # Display sensitivity results
    if sensitivity_results:
        sensitivity_df = pd.DataFrame(sensitivity_results)
        print("\nWeight Sensitivity Results:")
        print(sensitivity_df.round(2).to_string(index=False))
        
        # Visualize sensitivity
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        
        # Allocation distribution by scenario
        sensitivity_df.set_index('Scenario')[['China %', 'Mexico %', 'US %']].plot(
            kind='bar', stacked=True, ax=ax1,
            color=['#FF6B6B', '#4ECDC4', '#45B7D1']
        )
        ax1.set_title('Sourcing Allocation by Weight Scenario')
        ax1.set_ylabel('Allocation Percentage (%)')
        ax1.legend(title='Location')
        ax1.tick_params(axis='x', rotation=45)
        
        # Objective function values
        ax2.bar(sensitivity_df['Scenario'], sensitivity_df['Objective'], 
               color='#95A99C')
        ax2.set_title('Objective Function Value by Scenario')
        ax2.set_ylabel('Objective Value ($)')
        ax2.tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        plt.show()

    # Reset to original weights
    data.alpha = 0.6
    data.beta = 0.3
    data.gamma = 0.1